# 34 · Do independent segment-resolved datasets reproduce our positional findings?

Notebooks 31–33 found species-by-position differences along a reconstructed PT coordinate. Here
we test them against public data that measured PT segments **directly**.

| Dataset | Species | What was measured | Sex |
|---|---|---|---|
| Chen et al. 2021 JASN, GEO GSE150338 | mouse (C57BL/6) | Microdissected PTS1, PTS2, PTS3, bulk RNA-seq | male |
| Lee et al. 2015 JASN, GEO GSE56743 | rat (Sprague-Dawley) | Microdissected S1, S2, S3, bulk RNA-seq | male |
| CELLxGENE Census dataset 25818bf7 | mouse | snRNA (10x multiome), annotated PT segment 1/2/3, per donor | both |
| CELLxGENE Census dataset 09b518f9 (renal cortex benchmark) | human | snRNA (10x multiome), annotated convoluted PT versus PT segment 3, per donor | both |

No human study has microdissected S1/S2/S3, so human positional data come from snRNA annotations.
The human external comparison is **convoluted PT (S1+S2) versus S3**, and the same early/late
contrast is used for every dataset in the species comparison.

Three questions:
1. **Coordinate check.** Within each species, does our S1→S3 zonation match direct measurements?
2. **Species-by-position check.** Does (human late − early) − (mouse late − early) in the external
   snRNA data, measured on the same platform in both species, reproduce ours, gene by gene and
   pathway by pathway?
3. **Sex check.** Do female mice show the same late-PT mouse programs as male mice?

Public files live under `data/external/` (see `data/external/*/SOURCE.txt`).

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream33 = results_root / 'pt_pathway_registration_sensitivity'
external = data_root / 'external'
output = results_root / 'pt_external_segment_validation'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
required = [upstream33 / 'confident_pathways_with_robustness.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')
NOTEBOOK_LOGIC_VERSION = '34.external_segments.1'
MIN_CELLS = 50          # Minimum cells per donor and segment group in the snRNA pseudobulks.
print('Results folder:', output)

## 1 · Our per-specimen segment means

Rebuild notebook 12's structures and genes and average log-normalized expression within each
specimen and reviewed segment (S1, S2, S3). "Early" pools S1 and S2 structures (convoluted PT),
matching the human external annotation.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import gene_covariates
from pseudospace.pathways import build_pathway_membership

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
support = pt_obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_obs.shared_pseudospace.to_numpy(float)
pt_obs = pt_obs[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
adata = rebuild_pt_expression(pt_obs, {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                                       for v in expected.values() for n in v}, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['eligible'] = adata.var.measured_in_both_inputs.to_numpy() & universe.detection.ge(.02)
genes = pd.Index(universe.loc[universe.eligible, 'gene'])
Y = expression[:, universe.eligible.to_numpy()]
detection = universe.set_index('gene').loc[genes, ['detection_mouse', 'detection_human']]


def group_mean(mask):
    return np.asarray(Y[mask].mean(axis=0)).ravel()


ours = {}
for species, names in expected.items():
    for name in names:
        for seg in ('S1', 'S2', 'S3'):
            ours[(species, name, seg)] = group_mean((specimen == name) & (segment == seg))
        ours[(species, name, 'early')] = group_mean((specimen == name) & np.isin(segment, ['S1', 'S2']))
ours = pd.DataFrame(ours, index=genes)
ours.columns.names = ['species', 'specimen', 'segment']
species_mean = ours.T.groupby(level=['species', 'segment']).mean().T
print(f'{len(genes):,} genes; our segment means computed for {len(expected["mouse"]) + len(expected["human"])} specimens.')

## 2 · External segment profiles

- **Microdissection (bulk).** log2(TPM or RPKM + 1), averaged over replicates. Rat symbols are matched
  to our mouse symbols case-insensitively (an approximation; rat genes without a mouse symbol
  match are dropped).
- **snRNA (Census).** Per donor, segment and sex, raw counts summed over cells and divided by the
  cells' total counts over **all** genes (so normalization does not depend on our gene list), then
  log2(CPM + 1). Mouse embryonic donors are excluded. Donor-segment groups with fewer than
  `MIN_CELLS` cells are dropped, and a donor is kept only if it has every segment needed.

In [ ]:
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_bulk = mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
rat_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz', sep='\t')
rat_bulk = rat_bulk.drop_duplicates('Symbol').set_index('Symbol')


def bulk_segments(table, pattern):
    out = {}
    for seg in ('S1', 'S2', 'S3'):
        columns = [c for c in table.columns if pattern.format(seg=seg) in c]
        if not columns:
            raise ValueError(f'No columns for {seg}')
        out[seg] = np.log2(table[columns].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
    frame = pd.DataFrame(out)
    frame['early'] = frame[['S1', 'S2']].mean(axis=1)
    return frame


mouse_micro = bulk_segments(mouse_bulk, '_PT{seg}_').reindex(genes)
rat_micro = bulk_segments(rat_bulk, '{seg}_')
rat_micro.index = rat_micro.index.astype(str).str.upper()
rat_micro = rat_micro[~rat_micro.index.duplicated()].reindex(genes.str.upper())
rat_micro.index = genes


def census_segments(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)]
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    logcpm = np.log2(counts.loc[meta.index].div(meta.total_counts_all_genes, axis=0) * 1e6 + 1)
    rows = {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        values = {seg: logcpm.loc[block.index[block.segment.eq(seg)]].mean() for seg in needed}
        if early:
            # Pool the early segments by counts, not by averaging logs.
            pooled = counts.loc[block.index[block.segment.isin(early)]].sum()
            values['early'] = np.log2(pooled / meta.loc[block.index[block.segment.isin(early)], 'total_counts_all_genes'].sum() * 1e6 + 1)
        rows.update({(donor, sex, seg): v for seg, v in values.items()})
    frame = pd.DataFrame(rows).reindex(genes)
    frame.columns.names = ['donor', 'sex', 'segment']
    return frame, meta


mouse_sn, mouse_sn_meta = census_segments('mouse', {
    'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
    'epithelial cell of proximal tubule segment 3': 'S3'}, early=['S1', 'S2'])
human_sn, human_sn_meta = census_segments('human', {
    'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'},
    early=None)
donors = pd.DataFrame({
    'mouse snRNA': mouse_sn.columns.droplevel('segment').unique().to_frame(index=False).groupby('sex').size(),
    'human snRNA': human_sn.columns.droplevel('segment').unique().to_frame(index=False).groupby('sex').size()}).fillna(0).astype(int)
donors.to_csv(output / 'external_donors_by_sex.csv')
display(donors.rename_axis('donors kept'))

## 3 · Coordinate check: does our within-species zonation match direct measurements?

For each gene, the zonation contrast is late − early (S3 − S1/S2). We compare our contrast with
each external dataset by Spearman correlation across genes. Genes must be detected in at least 10%
of our structures in that species and expressed in the external data (mean log2 value ≥ 1 in
either segment group). A strong positive correlation means our reconstructed coordinate places
genes where direct segment measurements place them.

In [ ]:
from scipy.stats import spearmanr

ours_contrast = {sp: species_mean[(sp, 'S3')] - species_mean[(sp, 'early')] for sp in ('mouse', 'human')}
ours_s2 = {sp: species_mean[(sp, 'S2')] - species_mean[(sp, 'S1')] for sp in ('mouse', 'human')}


def contrast(frame, late='S3', early='early', sex=None):
    columns = frame.columns
    if sex is not None:
        frame = frame.loc[:, frame.columns.get_level_values('sex') == sex]
    late_mean = frame.xs(late, axis=1, level='segment').mean(axis=1)
    early_mean = frame.xs(early, axis=1, level='segment').mean(axis=1)
    return late_mean - early_mean, pd.concat([late_mean, early_mean], axis=1).max(axis=1)


external_contrasts = {
    'mouse microdissection (male)': ('mouse', mouse_micro.S3 - mouse_micro.early, mouse_micro[['S3', 'early']].max(axis=1)),
    'rat microdissection (male)': ('mouse', rat_micro.S3 - rat_micro.early, rat_micro[['S3', 'early']].max(axis=1)),
    'mouse snRNA, male donors': ('mouse', *contrast(mouse_sn, sex='male')),
    'mouse snRNA, female donors': ('mouse', *contrast(mouse_sn, sex='female')),
    'human snRNA, cortex': ('human', *contrast(human_sn)),
}
rows = []
for name, (sp, ext, level) in external_contrasts.items():
    keep = detection[f'detection_{sp}'].ge(.10) & level.ge(1) & ext.notna()
    rho = spearmanr(ours_contrast[sp][keep], ext[keep]).statistic
    strong = keep & ext.abs().ge(1)
    agree = np.mean(np.sign(ours_contrast[sp][strong]) == np.sign(ext[strong]))
    rows.append({'external dataset': name, 'our species': sp, 'genes': int(keep.sum()), 'Spearman rho': rho,
                 'genes with |external late−early| ≥ 1 log2': int(strong.sum()), 'direction agreement': agree})
coordinate_check = pd.DataFrame(rows)
coordinate_check.to_csv(output / 'coordinate_check.csv', index=False)
display(coordinate_check.round(3))
s2_check = spearmanr(ours_s2['mouse'], mouse_micro.S2 - mouse_micro.S1, nan_policy='omit').statistic
print(f'Mouse S2 − S1 contrast, ours versus microdissection: Spearman {s2_check:.3f}')

## 4 · Species-by-position check

The external interaction is (human late − early) − (mouse late − early). Both species come from 10x
multiome snRNA, so platform effects largely cancel within each species' contrast. We compare it
with our interaction across genes detected in both species, and focus on the genes our model calls
most position-dependent (top 10% by `T_spatial`). We repeat with female mice as the mouse
reference (sex check) and with microdissected male mouse (platform check).

In [ ]:
gene_stats = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv', index_col=0).reindex(genes)
ours_interaction = ours_contrast['human'] - ours_contrast['mouse']
human_ext, human_level = contrast(human_sn)
interaction_refs = {
    'human snRNA − male mouse snRNA': human_ext - contrast(mouse_sn, sex='male')[0],
    'human snRNA − female mouse snRNA': human_ext - contrast(mouse_sn, sex='female')[0],
    'human snRNA − male mouse microdissection': human_ext - (mouse_micro.S3 - mouse_micro.early),
}
both_detected = detection.min(axis=1).ge(.10) & human_level.ge(1)
top_spatial = gene_stats.T_spatial.ge(gene_stats.T_spatial.quantile(.9))
rows = []
for name, ext in interaction_refs.items():
    keep = both_detected & ext.notna()
    for subset, mask in (('all shared genes', keep), ('top 10% T_spatial', keep & top_spatial)):
        strong = mask & ext.abs().ge(1) & ours_interaction.abs().ge(.25)
        rows.append({'external interaction': name, 'genes': subset, 'n': int(mask.sum()),
                     'Spearman rho': spearmanr(ours_interaction[mask], ext[mask]).statistic,
                     'n strong in both': int(strong.sum()),
                     'direction agreement (strong)': np.mean(np.sign(ours_interaction[strong]) == np.sign(ext[strong]))})
interaction_check = pd.DataFrame(rows)
interaction_check.to_csv(output / 'species_by_position_check.csv', index=False)
display(interaction_check.round(3))
gene_level = pd.DataFrame({'our_interaction': ours_interaction, 'T_spatial': gene_stats.T_spatial,
                           **{name: ext for name, ext in interaction_refs.items()},
                           'detection_mouse': detection.detection_mouse, 'detection_human': detection.detection_human})
gene_level.to_csv(output / 'gene_level_interactions.csv')

## 5 · Pathway-level check

For each confident pathway (notebook 33), the median interaction over members detected in both
species, in our data and in the external male-mouse and female-mouse comparisons. The question is
whether the external data move each pathway in the same direction. Pathways with a median
external interaction near zero (|value| < 0.1) are counted as uninformative rather than discordant.

In [ ]:
import ast

coverage = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'pathway_coverage.csv').set_index('pathway_id')
members = {p: [g for g in ast.literal_eval(v) if g in set(genes)] for p, v in coverage.genes_present.dropna().items()}
confident = pd.read_csv(upstream33 / 'confident_pathways_with_robustness.csv')
rows = []
for row in confident.itertuples():
    member = [g for g in members.get(row.pathway_id, []) if both_detected.get(g, False)]
    if len(member) < 5:
        continue
    record = {'pathway_id': row.pathway_id, 'name': row.name, 'theme': row.theme, 'robustness': row.robustness,
              'core': bool(row.core), 'pseudospace_class': row.pseudospace_class, 'n_members_used': len(member),
              'ours': float(ours_interaction[member].median())}
    for name, ext in interaction_refs.items():
        record[name] = float(ext[member].median())
    rows.append(record)
pathway_level = pd.DataFrame(rows)


def concordance(frame, column):
    informative = frame[column].abs().ge(.1) & frame.ours.abs().ge(.1)
    agree = np.sign(frame.loc[informative, column]) == np.sign(frame.loc[informative, 'ours'])
    return int(informative.sum()), int(agree.sum())


summary_rows = []
for subset, mask in (('all confident', pathway_level.index == pathway_level.index),
                     ('robust', pathway_level.robustness.eq('robust')), ('core', pathway_level.core)):
    for name in interaction_refs:
        n, agree = concordance(pathway_level[mask], name)
        rho = spearmanr(pathway_level.loc[mask, 'ours'], pathway_level.loc[mask, name]).statistic
        summary_rows.append({'pathways': subset, 'external interaction': name, 'tested': int(mask.sum()),
                             'informative': n, 'same direction': agree, 'Spearman rho': rho})
pathway_check = pd.DataFrame(summary_rows)
pathway_level.to_csv(output / 'pathway_level_interactions.csv', index=False)
pathway_check.to_csv(output / 'pathway_level_check.csv', index=False)
display(pathway_check.round(3))

## 6 · Figure 10 · Our positional findings against direct segment measurements

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9,
                     'axes.titlesize': 10, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
HUMAN, MOUSE, INK = '#D55E00', '#0072B2', '#333333'


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=200)


fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9), layout='constrained')
for ax, (sp, ext_name) in zip(axes[:2], (('mouse', 'mouse microdissection (male)'), ('human', 'human snRNA, cortex'))):
    _, ext, level = external_contrasts[ext_name]
    keep = detection[f'detection_{sp}'].ge(.10) & level.ge(1) & ext.notna()
    ax.scatter(ext[keep], ours_contrast[sp][keep], s=4, alpha=.25, color=MOUSE if sp == 'mouse' else HUMAN,
               rasterized=True)
    row = coordinate_check[coordinate_check['external dataset'].eq(ext_name)].iloc[0]
    ax.axhline(0, color='#999999', lw=.6)
    ax.axvline(0, color='#999999', lw=.6)
    ax.set(xlabel=f'{ext_name}: S3 − early (log2)', ylabel=f'Ours, {sp}: S3 − early (log-norm)')
    ax.set_title(f'{sp.capitalize()} zonation · ρ = {row["Spearman rho"]:.2f}', loc='left')
ax = axes[2]
ext = interaction_refs['human snRNA − male mouse snRNA']
keep = both_detected & ext.notna()
ax.scatter(ext[keep & ~top_spatial], ours_interaction[keep & ~top_spatial], s=4, alpha=.2, color='#BBBBBB',
           rasterized=True, label='other shared genes')
ax.scatter(ext[keep & top_spatial], ours_interaction[keep & top_spatial], s=7, alpha=.6, color=INK,
           rasterized=True, label='top 10% T_spatial')
ax.axhline(0, color='#999999', lw=.6)
ax.axvline(0, color='#999999', lw=.6)
row = interaction_check[(interaction_check['external interaction'] == 'human snRNA − male mouse snRNA')
                        & interaction_check.genes.eq('top 10% T_spatial')].iloc[0]
ax.set(xlabel='External: (human − mouse) S3 − early, log2', ylabel='Ours: (human − mouse) S3 − early')
ax.set_title(f'Species × position · top genes ρ = {row["Spearman rho"]:.2f}', loc='left')
ax.legend(frameon=False, fontsize=7.5, loc='upper left')
save(fig, 'fig10_external_segment_validation')
plt.show()
plt.close(fig)

## 7 · Candidate genes, one row each

For the Tier 1 and Tier 2 genes of the literature check, the late − early contrast in every dataset.
"Replicates" = whether our contrast has the same sign as the external contrast. Values are log
scales of different platforms, so compare signs and rough magnitudes, not exact numbers.

In [ ]:
candidates = ['Gatm', 'Acadm', 'Hadh', 'Acaa2', 'Acsm3', 'Crot', 'Nudt19', 'Slc27a2', 'Ehhadh', 'Gclc', 'Gclm',
              'Hmgcs1', 'Cyp51', 'Slc22a6', 'Slc22a8', 'Slc13a3', 'Slc6a19', 'Dcxr', 'Igfbp4', 'Nt5e', 'Pah',
              'Glyat', 'Ugt3a1', 'Acox2', 'Cyp24a1', 'Phgdh', 'Psat1', 'Ugt1a9', 'Rbp4', 'Aox1', 'Pank1', 'Coasy',
              'Inmt', 'Me1', 'Lpl', 'Cyp2e1', 'Cyp7b1', 'Cyp4b1', 'Slc22a7']
candidates = [g for g in candidates if g in genes]
table = pd.DataFrame({
    'ours mouse': ours_contrast['mouse'], 'mouse micro (M)': mouse_micro.S3 - mouse_micro.early,
    'rat micro (M)': rat_micro.S3 - rat_micro.early, 'mouse sn (M)': contrast(mouse_sn, sex='male')[0],
    'mouse sn (F)': contrast(mouse_sn, sex='female')[0], 'ours human': ours_contrast['human'],
    'human sn': human_ext, 'ours interaction': ours_interaction,
    'external interaction (male mouse)': interaction_refs['human snRNA − male mouse snRNA'],
    'external interaction (female mouse)': interaction_refs['human snRNA − female mouse snRNA']}).loc[candidates]
table['interaction replicates (male ref)'] = np.sign(table['ours interaction']) == np.sign(table['external interaction (male mouse)'])
table['interaction replicates (female ref)'] = np.sign(table['ours interaction']) == np.sign(table['external interaction (female mouse)'])
table.to_csv(output / 'candidate_gene_external_profiles.csv')
display(table.round(2))
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'min_cells': MIN_CELLS,
            'donors': donors.to_dict(), 'coordinate_check': coordinate_check.round(4).to_dict(orient='records'),
            'species_by_position': interaction_check.round(4).to_dict(orient='records'),
            'pathway_level': pathway_check.round(4).to_dict(orient='records')}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))